# Lab 3: Probability Theory

> **Course:** Data Science Foundations
> **Lesson:** Day 3 Lab, companion to `3-probability-theory.md` and `3-probability-theory-notebook.ipynb`
> **Dataset:** `titanic` (seaborn). 891 passengers, with survival, sex, age, class and fare. Plus simulated dice.
> **Estimated time:** 90 minutes

## Lab Objectives

- Apply the **complement** and **addition** rules, and tell *mutually exclusive* apart from *independent*
- Compute marginal, joint, and conditional probabilities from real contingency tables
- Test independence from data and explain why $P(A \mid B) \ne P(B \mid A)$
- Use Bayes' theorem and the law of total probability, and verify both against direct counts
- Answer applied questions with `scipy.stats` and confirm each answer by **Monte Carlo simulation**
- Plot a **likelihood function** and find the maximum likelihood estimate

**How to work through this lab**

- Cells marked `# TODO` are yours to complete. Replace each `None` or comment with working code.
- Run every **Checkpoint** cell. If it prints `passed`, move on. If it raises an error, fix your code first.
- Each **Question** needs a short written answer in the "Your answer" cell below it.
- The dataset is different from the lesson on purpose. Adapt the lesson code; don't paste it.
- Worked solutions are in the `solutions/` folder. Attempt every part yourself before you open them: the struggle is where the learning happens.

## Part 0: Setup

In [ ]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

titanic = sns.load_dataset("titanic")
print(f"Shape: {titanic.shape}")
titanic[["survived", "sex", "age", "class", "fare", "embark_town"]].head()

## Part 1: The Basic Rules (extension)

Three rules the lesson used without naming:

| Rule | Formula |
|---|---|
| Complement | $P(\text{not } A) = 1 - P(A)$ |
| Addition (either event) | $P(A \cup B) = P(A) + P(B) - P(A \cap B)$ |
| Mutually exclusive | $A$ and $B$ cannot both happen, so $P(A \cap B) = 0$ |

Mutually exclusive is **not** the same as independent. If two events cannot happen together, knowing one happened tells you a lot about the other.

We roll two fair dice. All 36 outcomes are equally likely:

In [ ]:
outcomes = pd.DataFrame(list(itertools.product(range(1, 7), repeat=2)), columns=["die1", "die2"])
outcomes["total"] = outcomes["die1"] + outcomes["die2"]
print(f"{len(outcomes)} outcomes")
outcomes.head(8)

In [ ]:
# TODO 1.1: compute each probability as (number of matching outcomes) / 36
p_total_7 = None
p_not_7 = None              # use the complement rule, not counting
A = None                    # boolean Series: first die shows 6
B = None                    # boolean Series: total is at least 10
p_A_and_B = None
p_A_or_B_direct = None      # count outcomes in A OR B
p_A_or_B_rule = None        # P(A) + P(B) - P(A and B)

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert abs(p_total_7 - 1 / 6) < 1e-12 and abs(p_not_7 - 5 / 6) < 1e-12
assert abs(p_A_or_B_direct - p_A_or_B_rule) < 1e-12 and abs(p_A_or_B_rule - 9 / 36) < 1e-12
print("passed")

In [ ]:
# TODO 1.2: Monte Carlo check. Simulate 100,000 rolls of two dice with rng.integers(1, 7, size=(100_000, 2))
# and estimate P(total = 7) and P(A or B). How close are they to the exact answers?

**Question 1.** Are A and B mutually exclusive? Are they independent? Show the numbers.

**Your answer:**

*Write your answer here.*

## Part 2: Contingency Tables from Real Data

`pd.crosstab` builds the table. Its `normalize` argument turns counts into probabilities:

- `normalize="all"` gives **joint** probabilities (the whole table sums to 1)
- `normalize="index"` gives **conditional on the row** (each row sums to 1)
- `normalize="columns"` gives **conditional on the column** (each column sums to 1)

In [ ]:
# TODO 2.1: counts table of sex (rows) vs survived (columns), with margins=True
counts = None

# TODO 2.2: the same table as joint probabilities (no margins)
joint = None

# TODO 2.3: marginal probabilities
p_female = None
p_survived = None

# TODO 2.4: conditional probabilities
p_surv_given_female = None
p_surv_given_male = None
p_female_given_surv = None

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert abs(joint.values.sum() - 1) < 1e-9
assert abs(p_surv_given_female - joint.loc["female", 1] / p_female) < 1e-9
print("passed")

**Question 2.** Put $P(\text{survived} \mid \text{female})$ and $P(\text{female} \mid \text{survived})$ into plain English. Why are they different numbers?

**Your answer:**

*Write your answer here.*

## Part 3: Independence and Conditioning on More Than One Variable

In [ ]:
# TODO 3.1: P(survived | class) for each class, as a bar chart with a dashed line at the overall P(survived)

# TODO 3.2: independence check for class = First and survival:
# compare P(First and survived) with P(First) * P(survived)

In [ ]:
# TODO 3.3: survival rate for every (sex, class) combination as a heatmap (annot=True)
# Hint: titanic.pivot_table(index="sex", columns="class", values="survived", aggfunc="mean", observed=True)

**Question 3.** Is survival independent of class? What does the heatmap add that the sex-only and class-only tables could not show?

**Your answer:**

*Write your answer here.*

## Part 4: Bayes' Theorem on Real Data

$$P(A \mid B) = \frac{P(B \mid A)\,P(A)}{P(B)} \qquad\qquad P(B) = \sum_i P(B \mid A_i)\,P(A_i)$$

In [ ]:
# TODO 4.1: compute P(female | survived) with Bayes' theorem, using only
# p_surv_given_female, p_female and p_survived. Compare with the direct count from Part 2.
p_female_given_surv_bayes = None

# TODO 4.2: law of total probability. Rebuild P(survived) as sum over classes of P(survived | class) * P(class)
p_survived_total = None

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert abs(p_female_given_surv_bayes - p_female_given_surv) < 1e-9
assert abs(p_survived_total - p_survived) < 1e-9
print("passed")

### 4.3 Updating twice

Recall the lesson's medical test: 1% prevalence, 95% sensitivity, 5% false positive rate, so one positive result gives only a 16% chance of disease. What if the patient takes the test again (independently) and is positive again? **The posterior after the first test becomes the prior for the second.**

In [ ]:
def posterior(prior, sensitivity, false_pos_rate):
    # TODO: P(disease | positive) using Bayes' theorem and the law of total probability
    return None


# TODO: start from prior = 0.01 and apply posterior() 4 times in a row.
# Collect a table with columns: n_positive_tests, P_disease
updates = None
updates

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert abs(posterior(0.01, 0.95, 0.05) - 0.161) < 0.001
print("passed")

**Question 4.** How many consecutive positive tests does it take before the patient is more likely sick than healthy? Why do doctors order a confirmatory test instead of acting on the first one?

**Your answer:**

*Write your answer here.*

## Part 5: Applied Problems, Exact vs Simulated

Solve each problem with `scipy.stats`, then check it by simulating 200,000 draws with `rng`. Fill the `results` table.

1. **Delivery times** are Normal with mean 35 and sd 6 minutes. (a) P(a delivery takes more than 45 min)? (b) What delivery time should the company promise so that 95% of orders arrive on time?
2. **A clinic** receives patients at an average of 3.5 per hour (Poisson). (c) P(at most 2 patients in an hour)? (d) P(more than 10 patients in a *2-hour* window)? *Hint: the rate scales with the interval.*
3. **A quiz** has 20 multiple-choice questions with 4 options each. (e) P(a student who guesses every answer gets at least 10 right)?
4. **A shuttle** leaves at a uniformly random moment in a 15-minute window. (f) P(you wait between 5 and 8 minutes)?

In [ ]:
N = 200_000
results = {
    # "label": (exact_value, simulated_value)
    "a: P(delivery > 45)": (None, None),
    "b: 95th percentile delivery time": (None, None),
    "c: P(patients <= 2 in 1h)": (None, None),
    "d: P(patients > 10 in 2h)": (None, None),
    "e: P(quiz score >= 10 by guessing)": (None, None),
    "f: P(5 <= wait <= 8)": (None, None),
}
# TODO: replace every None, then show the table
pd.DataFrame(results, index=["exact", "simulated"]).T

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
for label, (exact, sim) in results.items():
    tol = 0.2 if label.startswith("b") else 0.005
    assert abs(exact - sim) < tol, label
print("passed")

**Question 5.** For (d) and (e), why does `sf(10)` answer "more than 10" but `sf(9)` answers "at least 10"? Which kind of distribution makes this detail matter?

**Your answer:**

*Write your answer here.*

## Part 6: Likelihood and Maximum Likelihood (extension)

The lesson's example: *you flip a coin 10 times and get 9 heads. How plausible is a fair coin?* The **likelihood** of a candidate heads-probability $p$ is the probability of the observed data if $p$ were true:

$$L(p) = P(\text{9 heads in 10} \mid p) = \binom{10}{9} p^9 (1-p)$$

The data is fixed; $p$ varies.

In [ ]:
p_grid = np.linspace(0, 1, 1001)

# TODO 6.1: likelihood of each p in p_grid (use stats.binom.pmf(9, 10, p_grid)) and plot it
likelihood = None

# TODO 6.2: the maximum likelihood estimate (the p with the highest likelihood)
p_mle = None

# TODO 6.3: how many times more likely is the data under p = 0.9 than under p = 0.5?
likelihood_ratio = None

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert abs(p_mle - 0.9) < 1e-9
print("passed")

**Question 6.** The area under the likelihood curve is about 0.09, not 1. Why is that fine? How is this connected to "training" a model?

**Your answer:**

*Write your answer here.*

## Part 7: Wrap-Up

A journalist writes: *"68% of Titanic survivors were women, so a woman had a 68% chance of surviving."* In 3 to 4 sentences, explain the mistake and give the correct figure.

**Your answer:**

*Write your answer here.*

> Submit your completed notebook to the Kanban board under your name by end of day.